# Afterimage preprocessing

Use the Python (afterimage_env) kernel. Choose a build in the first cell, check its inputs, then run the matching build cell. The cell streams progress and saves its log to D:.

## Configure paths and workers

In [ ]:
from pathlib import Path

ROOT = Path(r"D:\Afterimage_Experiments")
PROJECT = Path(r"C:\Users\moht2\Documents\Afterimage")
RAW_APTOS = Path(r"D:\aptos2019-blindness-detection")
SPLIT_MANIFEST = ROOT / "reference/aptos_split/dataset_manifest.json"
ARCHIVE_CONFIG = PROJECT / "experiments/artifacts/qwen_four_run_001/r16_clahe_on/resolved_configuration.json"

BUILD = "512_on"  # "512_on", "512_off", or "224_standard"
BUILD_TAG = "rebuild_01"
PARALLEL = True
WORKERS = 4
RUN_BUILD = False

## Shared setup

In [ ]:
import json
import sys

sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "code"))
from notebook_runner import run_command

print(f"Python: {sys.executable}")

SCRIPT = ROOT / "code/prepare_aptos.py"
TYPES = {"512_on": (512, "on"), "512_off": (512, "off"), "224_standard": (224, "off")}
size, clahe = TYPES[BUILD]
if PARALLEL and WORKERS < 1:
    raise ValueError("Use at least one worker or set PARALLEL = False.")
workers = WORKERS if PARALLEL else 0
build_dir = ROOT / "datasets" / f"aptos_{BUILD}_{BUILD_TAG}"
log_dir = ROOT / "logs/preprocessing" / build_dir.name
args = ["--aptos-root", RAW_APTOS, "--split-manifest", SPLIT_MANIFEST, "--archive-config", ARCHIVE_CONFIG,
        "--output", build_dir, "--size", size, "--clahe", clahe, "--workers", workers]

def build(kind):
    if RUN_BUILD and BUILD == kind:
        run_command(SCRIPT, args, log_dir / "build.log")
    else:
        print(f"Select BUILD = {kind!r} and set RUN_BUILD = True to start this cell.")

print(f"Selected {size} px, CLAHE {clahe}, {workers if workers > 1 else 'serial'}.")
print(f"New files: {build_dir}")

## Check the selected build

In [ ]:
run_command(SCRIPT, [*args, "--check-inputs"], log_dir / "preflight.log")

## Qwen with CLAHE

In [ ]:
build("512_on")

## Qwen without CLAHE

In [ ]:
build("512_off")

## RetiZero standard images at 224 pixels

In [ ]:
build("224_standard")

## View saved datasets and recent progress

In [ ]:
for folder in (ROOT / "datasets/aptos_512", ROOT / "datasets/aptos_224_standard", build_dir):
    manifest = folder / "dataset_manifest.json"
    if manifest.is_file():
        saved = json.loads(manifest.read_text(encoding="utf-8"))
        print(folder)
        for version, branches in saved["versions"].items():
            for branch, splits in branches.items():
                print(version, branch, {split: info["rows"] for split, info in splits.items()})
log = log_dir / "build.log"
if log.is_file():
    print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))